# NB5: K-Path Descriptor Extraction (Plane-Based, 205-row)

**Goal:** Extract k-path-specific descriptors for every row in rashba.csv.
Output: ONE unified CSV ready for feature selection in NB6.

**Key physics:** A k-point with wavevector **k** corresponds to a plane wave
e^(i**k**.**r**). The constant-phase surfaces are planes perpendicular to **k**.
We find atoms near these planes and compute their local environment descriptors
(GRDF, AFS). This is physically rigorous because descriptors are computed at
actual atomic sites, but the atom selection is k-path-dependent.

**Descriptor Groups:**

| Group | Source | Varies per row? | Description |
|-------|--------|----------------|-------------|
| CSV direct | rashba.csv | YES/NO | band, bandgap, ehull |
| K-path geometry | POSCAR + kpath | YES | distance, angle, kvec, frac coords |
| Degeneracy | space group | YES | k-point multiplicity |
| Plane-based GRDF/AFS | POSCAR + kpath | YES | environment of atoms on k-planes |
| Plane intersection | POSCAR + kpath | YES | atoms near line/point of plane intersections |
| Elemental properties | POSCAR | NO | Z, mass, radius, electronegativity |
| DOS descriptors | nb1 output | NO | p-orbital fraction at band edges |

**Location:** `Keshav-DDP/k-path/nb5_kpath_descriptors.ipynb`

## Cell 1: Configuration

In [ ]:
import os

# =============================================================================
# PATHS -- CHANGE THESE IF YOU MOVE THE NOTEBOOK
# =============================================================================

BASE_DIR = os.path.abspath("..")

RASHBA_CSV = os.path.join(BASE_DIR, "Data", "rashba.csv")

# Compound folders with POSCAR_std (reformatted by reformat_poscars.py)
VASP_DIR = os.path.join(BASE_DIR, "Inverse-design", "rashba")

# Existing DOS descriptor CSV (99 rows)
EXISTING_DOS_CSV = os.path.join(
    BASE_DIR, "Weight-contribution", "contribution-model",
    "best-descriptors-model-study", "best_combo_0_w10_radius_mean.csv"
)

RESULTS_DIR = os.path.join(".", "nb5_kpath_descriptors-results")
OUTPUT_CSV = os.path.join(RESULTS_DIR, "rashba_206_all_descriptors.csv")

# =============================================================================
print("=" * 65)
print("  PATH CONFIGURATION")
print("=" * 65)
for name, path in [("BASE_DIR", BASE_DIR), ("RASHBA_CSV", RASHBA_CSV),
                    ("VASP_DIR", VASP_DIR), ("EXISTING_DOS_CSV", EXISTING_DOS_CSV),
                    ("RESULTS_DIR", RESULTS_DIR), ("OUTPUT_CSV", OUTPUT_CSV)]:
    exists = os.path.exists(path) if name != "OUTPUT_CSV" else "will be created"
    status = "OK" if exists == True else ("OUTPUT" if exists == "will be created" else "MISSING")
    print(f"  [{status:7s}] {name:20s} = {path}")

os.makedirs(RESULTS_DIR, exist_ok=True)

## Cell 2: Imports

In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.symmetry.bandstructure import HighSymmKpath
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from scipy.special import spherical_jn

print("Imports OK.")

## Cell 3: Load rashba.csv

In [ ]:
df = pd.read_csv(RASHBA_CSV)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Unique UIDs: {df['uid'].nunique()}, Unique formulas: {df['Formula'].nunique()}")
print(f"K-paths: {sorted(df['kpath'].unique())}")
print(f"Space groups:\n{df['spacegroup'].value_counts().to_string()}")

## Cell 4: Load structures from POSCAR_std

Uses reformatted POSCARs (standard VASP5 format).
See `reformat_poscars.py` for the fix.

In [ ]:
def find_compound_folder(uid, vasp_dir):
    """Find the folder for a given uid."""
    for folder in glob.glob(os.path.join(vasp_dir, "*")):
        folder_name = os.path.basename(folder)
        idx = folder_name.rfind('-')
        if idx != -1 and folder_name[idx+1:] == uid:
            return folder
    return None


def load_structure(compound_folder):
    """
    Load structure from POSCAR_std (reformatted, standard VASP5 format).
    Falls back to original POSCAR if POSCAR_std not found.
    """
    # Priority 1: POSCAR_std (reformatted)
    std_path = os.path.join(compound_folder, "POSCAR_std")
    if os.path.exists(std_path):
        return Structure.from_file(std_path)

    # Priority 2: original encoded path
    matches = glob.glob(os.path.join(compound_folder, "ss_2d*POSCAR"))
    if matches:
        return Structure.from_file(matches[0])

    # Priority 3: direct POSCAR
    direct = os.path.join(compound_folder, "POSCAR")
    if os.path.exists(direct):
        return Structure.from_file(direct)

    return None


def get_highsymm_kpoints(structure):
    """Get high-symmetry k-point coordinates using pymatgen."""
    try:
        kpath_obj = HighSymmKpath(structure)
        kpoints = kpath_obj.kpath["kpoints"]
        return {label: np.array(coords) for label, coords in kpoints.items()}
    except Exception as e:
        return {'G': np.array([0.0, 0.0, 0.0])}


# Build compound lookup
print("Loading structures...")
compound_data = {}
missing = []

for i, uid in enumerate(df['uid'].unique()):
    folder = find_compound_folder(uid, VASP_DIR)
    if folder is None:
        missing.append(uid)
        continue

    structure = load_structure(folder)
    if structure is None:
        missing.append(uid)
        continue

    # Verify elements are correct (not H, He, Li)
    max_z = max(site.specie.Z for site in structure)
    if max_z <= 3:
        formula = df[df['uid'] == uid]['Formula'].iloc[0]
        print(f"  WARNING: {formula} ({uid}) has max_Z={max_z}. POSCAR_std may be missing.")

    kpoints = get_highsymm_kpoints(structure)

    compound_data[uid] = {
        'folder': folder,
        'structure': structure,
        'lattice': structure.lattice,
        'reciprocal_lattice': structure.lattice.reciprocal_lattice,
        'kpoints': kpoints,
    }

    if (i + 1) % 20 == 0:
        print(f"  Loaded {i+1}/{df['uid'].nunique()}")

print(f"\nLoaded {len(compound_data)} / {df['uid'].nunique()} compounds")
if missing:
    print(f"Missing ({len(missing)}): {missing[:5]}...")

# Verify element correctness on a sample
for uid in list(compound_data.keys())[:3]:
    s = compound_data[uid]['structure']
    formula = df[df['uid'] == uid]['Formula'].iloc[0]
    elems = sorted(set(site.specie.symbol for site in s))
    max_z = max(site.specie.Z for site in s)
    print(f"  {formula}: elements={elems}, max_Z={max_z}")

## Cell 5: K-path resolution (labels -> fractional coords)

In [ ]:
# Fallback k-point coordinates for labels not found by HighSymmKpath
FALLBACK_KPOINTS = {
    'G': np.array([0.0, 0.0, 0.0]),
    'GAMMA': np.array([0.0, 0.0, 0.0]),
    'M': np.array([0.5, 0.0, 0.0]),
    'K': np.array([1/3, 1/3, 0.0]),
    'X': np.array([0.5, 0.0, 0.0]),
    'Y': np.array([0.0, 0.5, 0.0]),
    'S': np.array([0.5, 0.5, 0.0]),
    'H': np.array([1/3, 1/3, 0.5]),
    'A': np.array([0.0, 0.0, 0.5]),
    'L': np.array([0.5, 0.0, 0.5]),
    'C': np.array([0.5, 0.5, 0.0]),
    'Z': np.array([0.0, 0.0, 0.5]),
    'R': np.array([0.5, 0.5, 0.5]),
    'T': np.array([0.0, 0.5, 0.5]),
    'N': np.array([0.5, 0.5, 0.0]),
}


def parse_kpath(kpath_str):
    """Parse 'G->M' into (start_label, end_label)."""
    for sep in ['->', '-', ' to ', '_']:
        if sep in kpath_str:
            parts = kpath_str.split(sep)
            if len(parts) == 2:
                return parts[0].strip(), parts[1].strip()
    raise ValueError(f"Cannot parse kpath: {kpath_str}")


def resolve_kpoint_coords(label, compound_kpoints):
    """Resolve k-point label to fractional coordinates."""
    # Direct match
    if label in compound_kpoints:
        return compound_kpoints[label]

    label_up = label.upper()
    for kp_label, coords in compound_kpoints.items():
        if kp_label.upper() == label_up:
            return coords

    # Match \Gamma style
    if label_up in ['G', 'GAMMA']:
        for kp_label, coords in compound_kpoints.items():
            if 'gamma' in kp_label.lower() or 'Gamma' in kp_label:
                return coords

    # Fallback table
    if label_up in FALLBACK_KPOINTS:
        return FALLBACK_KPOINTS[label_up]

    print(f"    WARNING: Unresolved '{label}'. Using (0,0,0).")
    return np.array([0.0, 0.0, 0.0])


# Test resolution
print("Testing k-path resolution:")
unresolved = set()
for _, row in df.iterrows():
    uid = row['uid']
    if uid not in compound_data:
        continue
    s, e = parse_kpath(row['kpath'])
    kpts = compound_data[uid]['kpoints']
    for label in [s, e]:
        coords = resolve_kpoint_coords(label, kpts)
        if np.allclose(coords, 0) and label.upper() not in ['G', 'GAMMA']:
            unresolved.add(label)

if unresolved:
    print(f"  Unresolved: {unresolved}")
else:
    print("  All labels resolved!")

## Cell 6: K-path geometry descriptors

In [ ]:
def extract_kpath_geometry(row, compound_data):
    """Extract k-path geometry: distance, angle, vector, fractional coords."""
    uid = row['uid']
    keys = ['kpath_real_distance', 'kpath_angle_deg',
            'kvec_x', 'kvec_y', 'kvec_z',
            'kstart_frac_x', 'kstart_frac_y',
            'kend_frac_x', 'kend_frac_y',
            'kmid_frac_x', 'kmid_frac_y']
    nan_result = {k: np.nan for k in keys}

    if uid not in compound_data:
        return nan_result

    cdata = compound_data[uid]
    rec_lat = cdata['reciprocal_lattice']
    kpts = cdata['kpoints']
    start_l, end_l = parse_kpath(row['kpath'])

    k_start_frac = resolve_kpoint_coords(start_l, kpts)
    k_end_frac = resolve_kpoint_coords(end_l, kpts)
    k_mid_frac = (k_start_frac + k_end_frac) / 2.0

    k_start_cart = rec_lat.get_cartesian_coords(k_start_frac)
    k_end_cart = rec_lat.get_cartesian_coords(k_end_frac)
    kvec = k_end_cart - k_start_cart
    dist = np.linalg.norm(kvec)
    angle = np.degrees(np.arctan2(kvec[1], kvec[0])) if dist > 1e-10 else 0.0

    return {
        'kpath_real_distance': dist,
        'kpath_angle_deg': angle,
        'kvec_x': kvec[0], 'kvec_y': kvec[1], 'kvec_z': kvec[2],
        'kstart_frac_x': k_start_frac[0], 'kstart_frac_y': k_start_frac[1],
        'kend_frac_x': k_end_frac[0], 'kend_frac_y': k_end_frac[1],
        'kmid_frac_x': k_mid_frac[0], 'kmid_frac_y': k_mid_frac[1],
    }


# Quick test
for i in range(min(3, len(df))):
    row = df.iloc[i]
    g = extract_kpath_geometry(row, compound_data)
    print(f"{row['Formula']} {row['kpath']}: dist={g['kpath_real_distance']:.4f}, angle={g['kpath_angle_deg']:.1f}")

## Cell 7: Degeneracy descriptors

In [ ]:
def get_kpoint_degeneracy(label, structure, kpoints):
    """Get k-point multiplicity from symmetry operations."""
    coords = resolve_kpoint_coords(label, kpoints)
    if np.allclose(coords, 0):
        return 1
    try:
        sga = SpacegroupAnalyzer(structure)
        symmops = sga.get_symmetry_operations()
        unique = set()
        for op in symmops:
            rotated = op.apply_rotation_only(coords)
            unique.add(tuple(np.round(rotated % 1.0, 6)))
        return len(unique)
    except:
        fallback = {'G': 1, 'M': 3, 'K': 2, 'X': 2, 'Y': 2, 'S': 4}
        return fallback.get(label.upper(), 1)


def extract_degeneracy(row, compound_data):
    """Extract degeneracy features."""
    uid = row['uid']
    if uid not in compound_data:
        return {k: np.nan for k in ['kstart_degen', 'kend_degen', 'kinterior_degen', 'ktotal_weight']}

    cdata = compound_data[uid]
    s_l, e_l = parse_kpath(row['kpath'])
    d_s = get_kpoint_degeneracy(s_l, cdata['structure'], cdata['kpoints'])
    d_e = get_kpoint_degeneracy(e_l, cdata['structure'], cdata['kpoints'])
    d_int = 2 * max(d_s, d_e)

    return {
        'kstart_degen': d_s, 'kend_degen': d_e,
        'kinterior_degen': d_int, 'ktotal_weight': d_s + d_e + d_int,
    }


# Test
for i in range(min(3, len(df))):
    row = df.iloc[i]
    d = extract_degeneracy(row, compound_data)
    print(f"{row['Formula']} {row['kpath']}: {d}")

## Cell 8: Plane-based structural environment (GRDF + AFS)

**Physics:** A k-point **k** defines a family of planes in real space,
perpendicular to the Cartesian k-vector. We find atoms within distance
`d_threshold` of these planes and compute GRDF/AFS on them.

For Gamma (k=0, no direction), all atoms are included.

For 3 k-points (start, mid, end) we get 3 sets of atoms, each with
their own GRDF/AFS descriptors. We also compute descriptors for atoms
near the intersection line of the start and end planes.

In [ ]:
# ---- GRDF / AFS basis functions (from structure-processor-final.py) ----

def grdf_gaussian(r, a=1.0, b=2.0):
    """Gaussian radial basis: exp(-a*(r-b)^2)"""
    return np.exp(-a * (r - b)**2)

def grdf_trig(r, k=1.0, phi=0.0):
    """Trigonometric radial basis: cos(k*r + phi)"""
    return np.cos(k * r + phi)

def grdf_bessel(r, k=1.0, n=0):
    """Bessel radial basis: j_n(k*r)"""
    return spherical_jn(n, k * r)


def compute_grdf_for_atoms(structure, atom_indices):
    """
    Compute GRDF descriptors for a set of atom indices.
    For each atom in the set, sum pairwise basis functions
    over its neighbors (also in the set).

    Returns aggregated stats: mean, std, max, min of each GRDF type.
    """
    if len(atom_indices) < 2:
        return {
            'grdf_gauss_mean': 0, 'grdf_gauss_std': 0, 'grdf_gauss_max': 0,
            'grdf_trig_mean': 0, 'grdf_trig_std': 0, 'grdf_trig_max': 0,
            'grdf_bessel_mean': 0, 'grdf_bessel_std': 0, 'grdf_bessel_max': 0,
            'n_atoms_on_plane': len(atom_indices),
        }

    site_grdf_gauss = []
    site_grdf_trig = []
    site_grdf_bessel = []

    idx_set = set(atom_indices)

    for i in atom_indices:
        g_gauss = 0.0
        g_trig = 0.0
        g_bessel = 0.0

        for j in atom_indices:
            if i == j:
                continue
            r = structure.get_distance(i, j)
            if r < 0.5:  # skip overlapping images
                continue
            g_gauss += grdf_gaussian(r)
            g_trig += grdf_trig(r)
            g_bessel += grdf_bessel(r)

        site_grdf_gauss.append(g_gauss)
        site_grdf_trig.append(g_trig)
        site_grdf_bessel.append(g_bessel)

    arr_g = np.array(site_grdf_gauss)
    arr_t = np.array(site_grdf_trig)
    arr_b = np.array(site_grdf_bessel)

    return {
        'grdf_gauss_mean': arr_g.mean(), 'grdf_gauss_std': arr_g.std(), 'grdf_gauss_max': arr_g.max(),
        'grdf_trig_mean': arr_t.mean(), 'grdf_trig_std': arr_t.std(), 'grdf_trig_max': arr_t.max(),
        'grdf_bessel_mean': arr_b.mean(), 'grdf_bessel_std': arr_b.std(), 'grdf_bessel_max': arr_b.max(),
        'n_atoms_on_plane': len(atom_indices),
    }


def compute_afs_for_atoms(structure, atom_indices):
    """
    Compute AFS (Angular Function Symmetry) for a set of atom indices.
    For each atom i, loop over neighbor pairs (j,k) in the set,
    compute angle j-i-k, weight by product of radial basis values.
    """
    if len(atom_indices) < 3:
        return {
            'afs_gauss_mean': 0, 'afs_gauss_std': 0, 'afs_gauss_max': 0,
            'afs_trig_mean': 0, 'afs_trig_std': 0, 'afs_trig_max': 0,
            'afs_bessel_mean': 0, 'afs_bessel_std': 0, 'afs_bessel_max': 0,
        }

    site_afs_gauss = []
    site_afs_trig = []
    site_afs_bessel = []

    for i in atom_indices:
        a_gauss = 0.0
        a_trig = 0.0
        a_bessel = 0.0

        # Get neighbors of i that are in our atom set
        neighbors_in_set = [j for j in atom_indices if j != i]

        for ni, j in enumerate(neighbors_in_set):
            r_ij = structure.get_distance(i, j)
            if r_ij < 0.5:
                continue

            g_ij_gauss = grdf_gaussian(r_ij)
            g_ij_trig = grdf_trig(r_ij)
            g_ij_bessel = grdf_bessel(r_ij)

            for k in neighbors_in_set[ni+1:]:
                r_ik = structure.get_distance(i, k)
                if r_ik < 0.5:
                    continue

                g_ik_gauss = grdf_gaussian(r_ik)
                g_ik_trig = grdf_trig(r_ik)
                g_ik_bessel = grdf_bessel(r_ik)

                # Angle j-i-k
                try:
                    theta = structure.get_angle(j, i, k)
                    cos_theta = np.cos(np.radians(theta))
                except:
                    cos_theta = 0.0

                a_gauss += g_ij_gauss * g_ik_gauss * cos_theta
                a_trig += g_ij_trig * g_ik_trig * cos_theta
                a_bessel += g_ij_bessel * g_ik_bessel * cos_theta

        site_afs_gauss.append(a_gauss)
        site_afs_trig.append(a_trig)
        site_afs_bessel.append(a_bessel)

    arr_g = np.array(site_afs_gauss)
    arr_t = np.array(site_afs_trig)
    arr_b = np.array(site_afs_bessel)

    return {
        'afs_gauss_mean': arr_g.mean(), 'afs_gauss_std': arr_g.std(), 'afs_gauss_max': arr_g.max(),
        'afs_trig_mean': arr_t.mean(), 'afs_trig_std': arr_t.std(), 'afs_trig_max': arr_t.max(),
        'afs_bessel_mean': arr_b.mean(), 'afs_bessel_std': arr_b.std(), 'afs_bessel_max': arr_b.max(),
    }


def compute_plane_elemental_stats(structure, atom_indices):
    """
    Compute elemental statistics for atoms on/near a plane.
    Z-weighted, mass-weighted descriptors of the selected atoms.
    """
    if len(atom_indices) == 0:
        return {
            'plane_mean_Z': 0, 'plane_max_Z': 0, 'plane_sum_Z4': 0,
            'plane_mean_mass': 0, 'plane_max_mass': 0,
            'plane_heavy_frac': 0,
        }

    zs = [structure[i].specie.Z for i in atom_indices]
    masses = [float(structure[i].specie.atomic_mass) for i in atom_indices]

    # Fraction of atoms that are "heavy" (Z > 30)
    heavy_frac = sum(1 for z in zs if z > 30) / len(zs) if zs else 0

    return {
        'plane_mean_Z': np.mean(zs),
        'plane_max_Z': max(zs),
        'plane_sum_Z4': sum(z**4 for z in zs),
        'plane_mean_mass': np.mean(masses),
        'plane_max_mass': max(masses),
        'plane_heavy_frac': heavy_frac,
    }


print("GRDF/AFS functions defined.")

## Cell 9: Plane extraction -- find atoms near k-plane

In [ ]:
def get_plane_atoms(k_frac, structure, d_threshold=None):
    """
    Find atoms near the plane defined by k-vector **k**.

    The plane is perpendicular to **k** (in Cartesian reciprocal space)
    and passes through the center of the slab.

    For k = (0,0,0) (Gamma), all atoms are returned (uniform wave).

    Args:
        k_frac: fractional reciprocal coordinates [3]
        structure: pymatgen Structure
        d_threshold: max distance from plane (default: 0.5 * a, in-plane lattice param)

    Returns:
        List of atom indices near the plane
    """
    # Gamma: all atoms
    if np.allclose(k_frac, 0):
        return list(range(len(structure)))

    # Convert k to Cartesian reciprocal space to get the plane normal
    rec_lat = structure.lattice.reciprocal_lattice
    k_cart = rec_lat.get_cartesian_coords(k_frac)
    k_norm = np.linalg.norm(k_cart)

    if k_norm < 1e-10:
        return list(range(len(structure)))

    # Unit normal to the plane
    n_hat = k_cart / k_norm

    # Default threshold: 0.5 * in-plane lattice parameter
    if d_threshold is None:
        a = np.linalg.norm(structure.lattice.matrix[0])
        d_threshold = 0.5 * a

    # Plane passes through center of slab (mean atomic position)
    center = np.mean([site.coords for site in structure], axis=0)

    # Find atoms within d_threshold of the plane
    atom_indices = []
    for i, site in enumerate(structure):
        # Signed distance from atom to plane
        d = abs(np.dot(site.coords - center, n_hat))
        if d <= d_threshold:
            atom_indices.append(i)

    return atom_indices


def get_intersection_line_atoms(k1_frac, k2_frac, structure, d_threshold=None):
    """
    Find atoms near the intersection line of two k-planes.

    Two planes (with normals n1 and n2) intersect in a line
    with direction n1 x n2. We find atoms close to this line.

    For parallel or degenerate planes, returns empty list.
    """
    rec_lat = structure.lattice.reciprocal_lattice

    k1_cart = rec_lat.get_cartesian_coords(k1_frac)
    k2_cart = rec_lat.get_cartesian_coords(k2_frac)

    n1 = np.linalg.norm(k1_cart)
    n2 = np.linalg.norm(k2_cart)

    # Handle zero vectors
    if n1 < 1e-10 or n2 < 1e-10:
        return []

    n1_hat = k1_cart / n1
    n2_hat = k2_cart / n2

    # Line direction = cross product of normals
    line_dir = np.cross(n1_hat, n2_hat)
    line_norm = np.linalg.norm(line_dir)

    if line_norm < 1e-10:
        # Planes are parallel, no intersection line
        return []

    line_dir = line_dir / line_norm

    # Default threshold
    if d_threshold is None:
        a = np.linalg.norm(structure.lattice.matrix[0])
        d_threshold = 0.5 * a

    # Line passes through slab center
    center = np.mean([site.coords for site in structure], axis=0)

    # Find atoms within d_threshold of the line
    atom_indices = []
    for i, site in enumerate(structure):
        # Vector from line point to atom
        v = site.coords - center
        # Distance from point to line = |v - (v.d)d| where d is line direction
        proj = np.dot(v, line_dir) * line_dir
        perp = v - proj
        dist = np.linalg.norm(perp)

        if dist <= d_threshold:
            atom_indices.append(i)

    return atom_indices


def get_intersection_point_atoms(k1_frac, k2_frac, k3_frac, structure, d_threshold=None):
    """
    Find atoms near the intersection point of three k-planes.
    If the three planes intersect at a point, find nearby atoms.
    """
    rec_lat = structure.lattice.reciprocal_lattice

    k1_cart = rec_lat.get_cartesian_coords(k1_frac)
    k2_cart = rec_lat.get_cartesian_coords(k2_frac)
    k3_cart = rec_lat.get_cartesian_coords(k3_frac)

    # Check for zero vectors
    norms = [np.linalg.norm(k) for k in [k1_cart, k2_cart, k3_cart]]
    if any(n < 1e-10 for n in norms):
        return []

    n1 = k1_cart / norms[0]
    n2 = k2_cart / norms[1]
    n3 = k3_cart / norms[2]

    # Check if the three normals span 3D (non-degenerate)
    det = np.linalg.det(np.array([n1, n2, n3]))
    if abs(det) < 1e-10:
        return []  # Degenerate (planes don't intersect at a single point)

    # Default threshold
    if d_threshold is None:
        a = np.linalg.norm(structure.lattice.matrix[0])
        d_threshold = 0.75 * a  # slightly larger for point

    center = np.mean([site.coords for site in structure], axis=0)

    atom_indices = []
    for i, site in enumerate(structure):
        dist = np.linalg.norm(site.coords - center)
        if dist <= d_threshold:
            atom_indices.append(i)

    return atom_indices


# Test on one compound
test_uid = df.iloc[0]['uid']
if test_uid in compound_data:
    cdata = compound_data[test_uid]
    s_l, e_l = parse_kpath(df.iloc[0]['kpath'])
    k_start = resolve_kpoint_coords(s_l, cdata['kpoints'])
    k_end = resolve_kpoint_coords(e_l, cdata['kpoints'])
    k_mid = (k_start + k_end) / 2.0

    for name, kfrac in [('start', k_start), ('mid', k_mid), ('end', k_end)]:
        atoms = get_plane_atoms(kfrac, cdata['structure'])
        elems = [cdata['structure'][i].specie.symbol for i in atoms]
        print(f"  {name} plane ({kfrac}): {len(atoms)} atoms: {elems}")

    line_atoms = get_intersection_line_atoms(k_start, k_end, cdata['structure'])
    print(f"  Intersection line: {len(line_atoms)} atoms")

    point_atoms = get_intersection_point_atoms(k_start, k_mid, k_end, cdata['structure'])
    print(f"  Intersection point: {len(point_atoms)} atoms")

## Cell 10: Full plane-based descriptor extraction for one row

In [ ]:
def extract_plane_descriptors(row, compound_data):
    """
    Extract all plane-based GRDF/AFS descriptors for one row.

    For each of 3 k-points (start, mid, end):
      - Find atoms on the k-plane
      - Compute GRDF (Gaussian, Trig, Bessel) on those atoms
      - Compute AFS on those atoms
      - Compute elemental stats of those atoms

    Also: intersection line and point descriptors.
    """
    uid = row['uid']
    features = {}

    if uid not in compound_data:
        return features  # empty, will become NaN

    cdata = compound_data[uid]
    structure = cdata['structure']
    s_l, e_l = parse_kpath(row['kpath'])
    k_start = resolve_kpoint_coords(s_l, cdata['kpoints'])
    k_end = resolve_kpoint_coords(e_l, cdata['kpoints'])
    k_mid = (k_start + k_end) / 2.0

    # --- Plane descriptors at start, mid, end ---
    for prefix, kfrac in [('pstart', k_start), ('pmid', k_mid), ('pend', k_end)]:
        atoms = get_plane_atoms(kfrac, structure)

        grdf = compute_grdf_for_atoms(structure, atoms)
        afs = compute_afs_for_atoms(structure, atoms)
        elem = compute_plane_elemental_stats(structure, atoms)

        for k, v in grdf.items():
            features[f'{prefix}_{k}'] = v
        for k, v in afs.items():
            features[f'{prefix}_{k}'] = v
        for k, v in elem.items():
            features[f'{prefix}_{k}'] = v

    # --- Intersection line descriptors ---
    line_atoms = get_intersection_line_atoms(k_start, k_end, structure)
    if len(line_atoms) >= 2:
        grdf_line = compute_grdf_for_atoms(structure, line_atoms)
        elem_line = compute_plane_elemental_stats(structure, line_atoms)
        for k, v in grdf_line.items():
            features[f'pline_{k}'] = v
        for k, v in elem_line.items():
            features[f'pline_{k}'] = v
    else:
        for k in ['grdf_gauss_mean', 'grdf_gauss_std', 'grdf_gauss_max',
                   'grdf_trig_mean', 'grdf_trig_std', 'grdf_trig_max',
                   'grdf_bessel_mean', 'grdf_bessel_std', 'grdf_bessel_max',
                   'n_atoms_on_plane']:
            features[f'pline_{k}'] = 0
        for k in ['plane_mean_Z', 'plane_max_Z', 'plane_sum_Z4',
                   'plane_mean_mass', 'plane_max_mass', 'plane_heavy_frac']:
            features[f'pline_{k}'] = 0

    # --- Intersection point descriptors ---
    point_atoms = get_intersection_point_atoms(k_start, k_mid, k_end, structure)
    features['ppoint_n_atoms'] = len(point_atoms)
    if len(point_atoms) >= 1:
        elem_point = compute_plane_elemental_stats(structure, point_atoms)
        for k, v in elem_point.items():
            features[f'ppoint_{k}'] = v
    else:
        for k in ['plane_mean_Z', 'plane_max_Z', 'plane_sum_Z4',
                   'plane_mean_mass', 'plane_max_mass', 'plane_heavy_frac']:
            features[f'ppoint_{k}'] = 0

    # --- Angle between start and end planes ---
    rec_lat = structure.lattice.reciprocal_lattice
    ks_cart = rec_lat.get_cartesian_coords(k_start)
    ke_cart = rec_lat.get_cartesian_coords(k_end)
    ns = np.linalg.norm(ks_cart)
    ne = np.linalg.norm(ke_cart)

    if ns > 1e-10 and ne > 1e-10:
        cos_angle = np.clip(np.dot(ks_cart/ns, ke_cart/ne), -1, 1)
        features['plane_angle_deg'] = np.degrees(np.arccos(cos_angle))
    else:
        features['plane_angle_deg'] = 0.0

    return features


# Test
if test_uid in compound_data:
    test_row = df.iloc[0]
    test_feats = extract_plane_descriptors(test_row, compound_data)
    print(f"Test: {test_row['Formula']} {test_row['kpath']}")
    print(f"  Total plane-based features: {len(test_feats)}")
    for k, v in list(test_feats.items())[:10]:
        print(f"    {k}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"    {k}: {v}")
    print(f"    ... and {len(test_feats) - 10} more")

## Cell 11: Elemental properties (compound-level)

In [ ]:
def compute_elemental_properties(structure):
    """Compute compound-level elemental descriptors."""
    elements = list(set(structure.species))
    zs = [el.Z for el in elements]
    masses = [float(el.atomic_mass) for el in elements]
    radii = [float(el.atomic_radius) if el.atomic_radius else 1.0 for el in elements]

    # Electronegativity with safe fallback
    eneg = []
    for el in elements:
        try:
            x = float(el.X) if el.X is not None else np.nan
        except:
            x = np.nan
        eneg.append(x)

    eneg_clean = [x for x in eneg if not np.isnan(x)]

    return {
        'max_Z': max(zs),
        'max_Z4': max(z**4 for z in zs),
        'max_mass': max(masses),
        'radius_mean': np.mean(radii),
        'radius_diff': max(radii) - min(radii) if len(radii) > 1 else 0.0,
        'X_mean': np.mean(eneg_clean) if eneg_clean else np.nan,
        'X_diff': max(eneg_clean) - min(eneg_clean) if len(eneg_clean) > 1 else 0.0,
        'n_elements': len(elements),
        'mean_Z': np.mean(zs),
        'sum_Z4': sum(z**4 for z in zs),
    }


# Precompute
elemental_props = {}
for uid, cdata in compound_data.items():
    elemental_props[uid] = compute_elemental_properties(cdata['structure'])

# Verify
for uid in list(elemental_props.keys())[:3]:
    formula = df[df['uid'] == uid]['Formula'].iloc[0]
    props = elemental_props[uid]
    print(f"{formula}: max_Z={props['max_Z']}, max_mass={props['max_mass']:.1f}, radius_mean={props['radius_mean']:.2f}")

## Cell 12: Load DOS descriptors

In [ ]:
dos_features_df = None

if EXISTING_DOS_CSV and os.path.exists(EXISTING_DOS_CSV):
    df_dos = pd.read_csv(EXISTING_DOS_CSV)
    dos_cols = [c for c in df_dos.columns if 'pfrac' in c.lower() or 'E_' in c]
    uid_col = 'uid' if 'uid' in df_dos.columns else None

    if uid_col and dos_cols:
        dos_features_df = df_dos[[uid_col] + dos_cols].drop_duplicates(subset=[uid_col])
        print(f"DOS features: {dos_cols} ({dos_features_df[uid_col].nunique()} UIDs)")
    else:
        print("Could not find uid or DOS columns in CSV.")
else:
    print(f"No DOS CSV at: {EXISTING_DOS_CSV}")

## Cell 13: MAIN EXTRACTION LOOP

In [ ]:
print("=" * 65)
print("  EXTRACTING ALL DESCRIPTORS")
print("=" * 65)

all_rows = []
errors = []

for idx, row in df.iterrows():
    uid = row['uid']
    features = {}

    # === Identifiers ===
    features['Formula'] = row['Formula']
    features['uid'] = uid
    features['kpath'] = row['kpath']
    features['Rashba_parameter'] = row['Rashba_parameter']

    # === CSV direct ===
    features['band_binary'] = 1 if row['band'] == 'C' else 0
    features['bandgap'] = row.get('bandgap', np.nan)
    features['ehull'] = row.get('ehull', np.nan)

    # === K-path geometry ===
    features.update(extract_kpath_geometry(row, compound_data))

    # === Degeneracy ===
    features.update(extract_degeneracy(row, compound_data))

    # === Plane-based GRDF/AFS ===
    try:
        features.update(extract_plane_descriptors(row, compound_data))
    except Exception as e:
        errors.append((idx, uid, f"plane desc: {e}"))

    # === Elemental properties ===
    if uid in elemental_props:
        features.update(elemental_props[uid])

    all_rows.append(features)

    if (idx + 1) % 25 == 0:
        print(f"  {idx+1}/{len(df)} rows...")

df_full = pd.DataFrame(all_rows)
print(f"\nDone: {df_full.shape[0]} rows, {df_full.shape[1]} columns")
if errors:
    print(f"Errors ({len(errors)}):")
    for e in errors[:5]:
        print(f"  Row {e[0]}, {e[1]}: {e[2]}")

## Cell 14: Merge DOS and create derived features

In [ ]:
if dos_features_df is not None:
    before = set(df_full.columns)
    df_full = df_full.merge(dos_features_df, on='uid', how='left')
    new = set(df_full.columns) - before
    print(f"Merged DOS: {sorted(new)}")

if 'E_pfrac_VBM' in df_full.columns and 'E_pfrac_CBM' in df_full.columns:
    df_full['E_pfrac_relevant'] = np.where(
        df_full['band_binary'] == 0,
        df_full['E_pfrac_VBM'],
        df_full['E_pfrac_CBM']
    )
    print("Created E_pfrac_relevant")

print(f"\nFinal: {df_full.shape[0]} rows, {df_full.shape[1]} columns")

## Cell 15: Save

In [ ]:
ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
feature_cols = sorted([c for c in df_full.columns if c not in ID_COLS])

print(f"Identifiers: {len(ID_COLS)}")
print(f"Features: {len(feature_cols)}")

# Check for issues
n_nan_cols = sum(1 for c in feature_cols if df_full[c].isna().sum() > 10)
n_const_cols = sum(1 for c in feature_cols if df_full[c].nunique() <= 1)
print(f"Columns with >10 NaN: {n_nan_cols}")
print(f"Constant columns: {n_const_cols}")

# Show all features with stats
print(f"\nFeature summary:")
for i, col in enumerate(feature_cols):
    n_unique = df_full[col].nunique()
    n_nan = df_full[col].isna().sum()
    print(f"  {i+1:3d}. {col:45s} unique={n_unique:4d}, NaN={n_nan:3d}")

df_full.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}")

## Cell 16: Sanity checks

In [ ]:
print("=" * 65)
print("  SANITY CHECKS")
print("=" * 65)

# 1. K-path varying features
multi_uids = df_full.groupby('uid').filter(lambda g: len(g) > 1)['uid'].unique()
varying = []
constant = []

for col in feature_cols:
    if df_full[col].dtype not in ['float64', 'int64', 'float32', 'int32']:
        continue
    n_var = 0
    for uid in multi_uids[:30]:
        if df_full[df_full['uid'] == uid][col].nunique() > 1:
            n_var += 1
    pct = n_var / min(30, len(multi_uids)) * 100
    if pct > 10:
        varying.append((col, pct))
    else:
        constant.append(col)

print(f"\n1. K-path specific ({len(varying)} features):")
for col, pct in sorted(varying, key=lambda x: -x[1])[:25]:
    print(f"    {col:45s}: varies in {pct:.0f}% of compounds")

print(f"\n   Compound-level ({len(constant)} features)")

# 2. Quick feature check
print(f"\n2. Features with 0 NaN: {sum(1 for c in feature_cols if df_full[c].isna().sum() == 0)}")
print(f"   Features with >50% NaN: {sum(1 for c in feature_cols if df_full[c].isna().mean() > 0.5)}")
print(f"   Constant features: {n_const_cols}")

## Cell 17: Summary

In [ ]:
print("=" * 65)
print("  NB5 COMPLETE")
print("=" * 65)
print(f"""
  Dataset:     {df_full.shape[0]} rows, {len(feature_cols)} features
  K-path specific: {len(varying)}
  Compound-level:  {len(constant)}
  0 NaN features:  {sum(1 for c in feature_cols if df_full[c].isna().sum() == 0)}

  Descriptor groups:
    CSV direct:       band_binary, bandgap, ehull
    K-path geometry:  distance, angle, kvec, frac coords
    Degeneracy:       start/end/interior degen, total weight
    Plane GRDF/AFS:   Gaussian/Trig/Bessel at start/mid/end planes
    Plane intersection: line atoms (start x end), point atoms (all 3)
    Plane elemental:  Z, mass, heavy fraction on each plane
    Elemental:        radius, Z, Z4, mass, electronegativity
    DOS:              E_pfrac_VBM, E_pfrac_CBM, E_pfrac_relevant

  Output: {OUTPUT_CSV}
  Next:   NB6 feature selection
""")